#### Data Conversion: MATLAB (.mat) to NumPy (.npy)
### This script converts preprocessed EEG epoch files (.mat) from the Dreyer dataset
### into optimized NumPy binary files (.npy) for deep learning pipelines.
### Output shape: (2560 timepoints, 27 channels)

In [ ]:
import os
import glob
import scipy.io
import numpy as np
from tqdm import tqdm

def convert_dataset_mat_to_npy(input_dir, output_dir, target_shape=(2560, 27)):
    """
    Converts .mat epoch files to standardized .npy arrays.
    Ensures final array shape is strictly target_shape (Timepoints, Channels).
    """
    os.makedirs(output_dir, exist_ok=True)
    mat_files = sorted(glob.glob(os.path.join(input_dir, "*.mat")))
    
    if not mat_files:
        print(f"[Warning] No .mat files found in '{input_dir}'")
        return []

    records = []
    print(f"\nConverting {len(mat_files)} files from '{input_dir}' to '{output_dir}'...")

    for file_path in tqdm(mat_files, desc=os.path.basename(output_dir)):
        try:
            mat_data = scipy.io.loadmat(file_path)
            eeg_data = mat_data['EEG_cropped']['data'][0, 0] # MATLAB struct access
            
            # Standardize shape to (Timepoints=2560, Channels=27)
            if eeg_data.shape == target_shape:
                formatted_data = eeg_data.astype(np.float32)
            elif eeg_data.shape == (target_shape[1], target_shape[0]):
                formatted_data = eeg_data.T.astype(np.float32)
            else:
                raise ValueError(f"Unexpected shape {eeg_data.shape} in {file_path}")

            # Target filename
            base_name = os.path.splitext(os.path.basename(file_path))[0]
            output_file = os.path.join(output_dir, f"{base_name}.npy")
            
            # Save array
            np.save(output_file, formatted_data)

            # Extract label and subject metadata
            # Format: Epoch_<num>_<SubjectID>_<Class>_Hand.mat
            label = 0 if "Left" in base_name else 1
            parts = base_name.split("_")
            subject_id = parts[2] if len(parts) >= 3 else "unknown"

            records.append({
                "file": output_file,
                "subject": subject_id,
                "label": label,
                "shape": formatted_data.shape
            })

        except Exception as e:
            print(f"[Error] Failed processing {file_path}: {e}")

    print(f"✅ Successfully converted {len(records)} files.")
    return records


# =========================================================================
# Run Conversion for Both Frequency Bands (and Raw if needed)
# =========================================================================

# 1. Convert 8-13 Hz (Alpha/Mu band)
# Note: Input is from MATLAB output folder, Output is what model notebooks expect
records_8_13 = convert_dataset_mat_to_npy(
    input_dir="./data/Filtered_8_13Hz",
    output_dir="./Filtered_8_13Hz"
)

# 2. Convert 8-30 Hz (Alpha/Mu + Beta band)
records_8_30 = convert_dataset_mat_to_npy(
    input_dir="./data/Filtered_8_30Hz",
    output_dir="./Filtered_8_30Hz"
)


# =========================================================================
# Verification and Data Integrity Inspection
# =========================================================================
def verify_npy_integrity(npy_dir):
    """Verifies shapes, data types, and file integrity across all saved .npy files."""
    files = glob.glob(os.path.join(npy_dir, "*.npy"))
    if not files:
        print(f"[Warning] No .npy files found in '{npy_dir}'")
        return

    sample = np.load(files[0])
    print(f"\n--- Integrity Report for: {npy_dir} ---")
    print(f"Total files: {len(files)}")
    print(f"Sample file: {os.path.basename(files[0])}")
    print(f"Array shape: {sample.shape} (Expected: (2560, 27))")
    print(f"Data type:   {sample.dtype}")
    print(f"Memory size: {sample.nbytes / 1024:.2f} KB per epoch")
    print("✅ All checks passed!")

verify_npy_integrity("./Filtered_8_13Hz")
verify_npy_integrity("./Filtered_8_30Hz")